In [12]:
import json
import os
import cv2
from copy import deepcopy

def extract_helmet_data(original_annotation_file, image_folder, output_folder, 
                        twowheeler_category_id=3, helmet_category_id=1, no_helmet_category_id=2):
    """
    Extracts two-wheeler images and helmet annotations while handling No Helmet cases correctly.
    """

    os.makedirs(output_folder, exist_ok=True)
    
    with open(original_annotation_file, "r") as f:
        coco_data = json.load(f)
    
    annotations = coco_data["annotations"]
    images = {img["id"]: img for img in coco_data["images"]}
    
    helmet_annotations = []
    new_images = []
    new_ann_id = 0  # Unique annotation ID tracker
    
    for img_data in coco_data["images"]:
        img_id = img_data["id"]
        img_path = os.path.join(image_folder, img_data["file_name"])
        
        if not os.path.exists(img_path):
            print(f"Warning: {img_path} not found!")
            continue
        
        img = cv2.imread(img_path)
        img_h, img_w = img.shape[:2]  # Get image dimensions
        
        twowheelers = [ann for ann in annotations if ann["image_id"] == img_id and ann["category_id"] == twowheeler_category_id]
        helmets = [ann for ann in annotations if ann["image_id"] == img_id and ann["category_id"] == helmet_category_id]
        
        for i, tw_ann in enumerate(twowheelers):
            x, y, w, h = map(int, tw_ann["bbox"])
            cropped_img = img[y:y+h, x:x+w]
            new_img_id = img_id * 100 + i  # Ensures unique integer IDs

            helmet_in_tw = []
            for ann in helmets:
                ax, ay, aw, ah = map(int, ann["bbox"])
                
                # Check if the helmet overlaps with the two-wheeler region
                if (ax < (x + w) and (ax + aw) > x) and (ay < (y + h) and (ay + ah) > y):
                    # Adjust helmet bbox relative to cropped image
                    new_bbox = [max(0, ax - x), max(0, ay - y), aw, ah]
                    new_ann = deepcopy(ann)
                    new_ann["image_id"] = new_img_id
                    new_ann["bbox"] = new_bbox
                    new_ann["id"] = new_ann_id
                    new_ann["category_id"] = 0  # ✅ Helmet class ID fixed
                    new_ann_id += 1
                    helmet_in_tw.append(new_ann)
            
            # Save cropped two-wheeler image
            helmet_path = os.path.join(output_folder, f"{new_img_id}.jpg")
            cv2.imwrite(helmet_path, cropped_img)
            new_images.append({
                "id": new_img_id, 
                "file_name": f"{new_img_id}.jpg", 
                "width": w, "height": h
            })
            
            if not helmet_in_tw:
                # Explicitly label this as "No Helmet" (Full frame bbox)
                helmet_annotations.append({
                    "id": new_ann_id,
                    "image_id": new_img_id,
                    "category_id": 1,  # ✅ No Helmet class ID fixed
                    "bbox": [0, 0, w, h],  # Full frame
                    "area": w * h,
                    "iscrowd": 0
                })
                new_ann_id += 1
            else:
                helmet_annotations.extend(helmet_in_tw)
    
    helmet_coco_data = {
        "images": new_images,
        "annotations": helmet_annotations,
        "categories": [
            {"id": 0, "name": "Helmet"},      # ✅ Fixed class IDs
            {"id": 1, "name": "No Helmet"}
        ]
    }
    
    with open(os.path.join(output_folder, "_annotations.json"), "w") as f:
        json.dump(helmet_coco_data, f, indent=4)
    
    print(f"Processing complete! Data saved in '{output_folder}'.")


In [13]:
extract_helmet_data(
    original_annotation_file="Data/train/_annotations.coco.json",
    image_folder="Data/train",
    output_folder="Data_helmet_app2/train"
)

Processing complete! Data saved in 'Data_helmet_app2/train'.


In [14]:
extract_helmet_data(
    original_annotation_file="Data/test/_annotations.coco.json",
    image_folder="Data/test",
    output_folder="Data_helmet_app2/test"
)

Processing complete! Data saved in 'Data_helmet_app2/test'.


In [15]:
extract_helmet_data(
    original_annotation_file="Data/valid/_annotations.coco.json",
    image_folder="Data/valid",
    output_folder="Data_helmet_app2/valid"
)

Processing complete! Data saved in 'Data_helmet_app2/valid'.
